# Malaria Burden & Intervention Effectiveness Analysis
## Notebook 01 — Data Preparation & SQL Pipeline

**Author:** Stephen Udoh
**Date:** June 2026
**GitHub:** github.com/Stephen-Udoh/malaria-analysis

---

### Research Question
Does ITN coverage significantly reduce malaria incidence 
and mortality across sub-Saharan African countries? 
Which demographic and geographic factors moderate 
this relationship?

---

### Data Sources
| Source | Dataset | Years |
|--------|---------|-------|
| WHO GHO | Malaria burden — incidence & mortality | 2000–2024 |
| WHO GHO | Malaria interventions — ITN, IRS, ACT | 2015–2024 |
| World Bank | GDP per capita & urban population % | 2000–2024 |

---

### Notebook Structure
1. Environment setup and database connection
2. Load raw CSV files into MySQL
3. Audit tables in SQL
4. Clean and filter to sub-Saharan Africa
5. Transform World Bank data from wide to long
6. Convert counts to rates
7. Join all datasets into analytical tables
8. Export final datasets for analysis

---

### Analytical Plan
| Analysis | Period | Method | Tool |
|----------|--------|--------|------|
| Burden trend over time | 2000–2024 | Time series | Python |
| Intervention effectiveness | 2015–2024 | Regression | SPSS + Python |
| ITN scale-up comparison | 2015–2024 | T-test | SPSS + Python |
| Burden prediction model | 2015–2024 | Random Forest | Python ML |
| Malaria forecast to 2030 | 2000–2024 | Prophet | Python ML |

## 1. Environment Setup & Database Connection

In [1]:
# =============================================================
# IMPORTS
# All libraries used in this notebook declared in one place
# Add new libraries here rather than scattered across cells
# =============================================================

import pandas as pd                         # data manipulation
import numpy as np                          # numerical operations
from sqlalchemy import create_engine, text  # database connection
from dotenv import load_dotenv              # load credentials
import os                                   # access environment variables
import warnings                             # suppress minor warnings

warnings.filterwarnings('ignore')

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


### 1.1 Load Credentials & Configure Connection
Credentials are stored in a `.env` file in the project root.
This file is excluded from version control via `.gitignore`.
Anyone reusing this notebook creates their own `.env` file.

In [2]:
# =============================================================
# LOAD CREDENTIALS FROM .env FILE
# Reads database credentials from local environment file
# Never hardcodes sensitive information in the notebook
# =============================================================

# Build path to .env file in project root
# Our notebook is in /notebooks/ so we go one level up
dotenv_path = os.path.join(os.path.dirname(os.getcwd()), '.env')

# Load the environment variables from .env file
load_dotenv(dotenv_path)

# Read each credential from environment
DB_CONFIG = {
    'user'    : os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'host'    : os.getenv('DB_HOST'),
    'port'    : os.getenv('DB_PORT'),
    'database': os.getenv('DB_NAME')
}

# Verify all credentials loaded — catches missing .env entries
missing = [key for key, val in DB_CONFIG.items() if val is None]

if missing:
    print(f"✗ Missing credentials: {missing}")
    print("  Check your .env file exists in project root")
else:
    print("✓ All credentials loaded from .env file")
    print(f"  User     : {DB_CONFIG['user']}")
    print(f"  Host     : {DB_CONFIG['host']}")
    print(f"  Port     : {DB_CONFIG['port']}")
    print(f"  Database : {DB_CONFIG['database']}")
    print("  Password : ********")

✓ All credentials loaded from .env file
  User     : root
  Host     : localhost
  Port     : 3306
  Database : malaria_project
  Password : ********


### 1.2 Database Connection

In [3]:
# Build connection string from loaded credentials
CONNECTION_STRING = (
    f"mysql+mysqlconnector://"
    f"{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/"
    f"{DB_CONFIG['database']}"
)

def create_db_engine(connection_string):
    """
    Creates and validates a SQLAlchemy engine.
    Returns engine object if successful, None if failed.
    """
    try:
        engine = create_engine(connection_string)
        with engine.connect() as conn:
            result = conn.execute(text("SELECT DATABASE()"))
            db_name = result.fetchone()[0]
            print(f"✓ Connected to: {db_name}")
        return engine
    except Exception as e:
        print(f"✗ Connection failed: {e}")
        return None

engine = create_db_engine(CONNECTION_STRING)

✓ Connected to: malaria_project


In [4]:
def run_query(query, engine):
    """
    Executes a SQL query and returns a pandas DataFrame.
    Used throughout this notebook for all SQL operations.
    """
    try:
        with engine.connect() as conn:
            return pd.read_sql(text(query), conn)
    except Exception as e:
        print(f"✗ Query failed: {e}")
        return None

## 2. Load Raw CSV Files into MySQL

Each WHO file is in long format — one row per country per year.
World Bank files are in wide format — years as columns.
All files are loaded as-is into raw tables first.
Cleaning and transformation happens in subsequent steps.

In [5]:
import os

# Path to raw data folder — one level up from notebooks
RAW_DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'raw')

# Map each file to a meaningful table name in MySQL
# Keys = table names, Values = CSV filenames
FILES_TO_LOAD = {
    'raw_mortality'     : 'MALARIA_EST_MORTALITY.csv',
    'raw_incidence'     : 'MALARIA_EST_INCIDENCE.csv',
    'raw_itn'           : 'MALARIA_ITN_COVERAGE.csv',
    'raw_act'           : 'MALARIA_ACT_TREATED.csv',
    'raw_irs'           : 'MALARIA_IRS_COVERAGE.csv',
    'raw_gdp'           : 'API_NY.GDP.PCAP.CD_DS2_en_csv_v2_273495.csv',
    'raw_urban'         : 'API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_276471.csv'
}

print(f"Data path: {RAW_DATA_PATH}")
print(f"Files to load: {len(FILES_TO_LOAD)}")

Data path: C:\Users\HP\Desktop\malaria_analysis\data\raw
Files to load: 7


In [6]:
def load_csv_to_mysql(files_dict, data_path, engine):
    """
    Loads multiple CSV files into MySQL as raw tables.
    
    Replaces table if it already exists — safe to re-run.
    Skips files that cannot be found and reports clearly.
    Returns a summary of what was loaded successfully.
    """
    summary = []

    for table_name, filename in files_dict.items():
        filepath = os.path.join(data_path, filename)

        # Check file exists before attempting load
        if not os.path.exists(filepath):
            print(f"✗ File not found: {filename}")
            summary.append({
                'table'  : table_name,
                'file'   : filename,
                'rows'   : 0,
                'status' : 'File not found'
            })
            continue

        try:
            # Read CSV into pandas
            df = pd.read_csv(filepath, encoding='utf-8')

            # Write to MySQL — replace if table exists
            df.to_sql(
                name      = table_name,
                con       = engine,
                if_exists = 'replace',
                index     = False
            )

            print(f"✓ {table_name:<20} {len(df):>6} rows loaded")
            summary.append({
                'table'  : table_name,
                'file'   : filename,
                'rows'   : len(df),
                'status' : 'Success'
            })

        except Exception as e:
            print(f"✗ {table_name} failed: {e}")
            summary.append({
                'table'  : table_name,
                'file'   : filename,
                'rows'   : 0,
                'status' : str(e)
            })

    # Return summary as DataFrame for easy review
    return pd.DataFrame(summary)


# Run the loader
print("Loading files into MySQL...\n")
load_summary = load_csv_to_mysql(FILES_TO_LOAD, RAW_DATA_PATH, engine)

print("\n--- Load Summary ---")
print(load_summary)

Loading files into MySQL...

✓ raw_mortality          2834 rows loaded
✓ raw_incidence          2855 rows loaded
✓ raw_itn                 400 rows loaded
✓ raw_act                 505 rows loaded
✓ raw_irs                 579 rows loaded
✓ raw_gdp                 266 rows loaded
✓ raw_urban               266 rows loaded

--- Load Summary ---
           table                                            file  rows  \
0  raw_mortality                       MALARIA_EST_MORTALITY.csv  2834   
1  raw_incidence                       MALARIA_EST_INCIDENCE.csv  2855   
2        raw_itn                        MALARIA_ITN_COVERAGE.csv   400   
3        raw_act                         MALARIA_ACT_TREATED.csv   505   
4        raw_irs                        MALARIA_IRS_COVERAGE.csv   579   
5        raw_gdp     API_NY.GDP.PCAP.CD_DS2_en_csv_v2_273495.csv   266   
6      raw_urban  API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_276471.csv   266   

    status  
0  Success  
1  Success  
2  Success  
3  Success

## 3. Verify Tables in MySQL

Confirm all seven raw tables loaded correctly.
Check row counts match our audit log expectations.

In [7]:
# Verify all tables exist in the database
# Row counts should match the audit log exactly

verification_query = """
    SELECT 
        table_name,
        table_rows
    FROM information_schema.tables
    WHERE table_schema = 'malaria_project'
    ORDER BY table_name;
"""

tables = run_query(verification_query, engine)
print("Tables in malaria_project database:\n")
print(tables.to_string(index=False))

Tables in malaria_project database:

     TABLE_NAME  TABLE_ROWS
      clean_act         271
      clean_gdp        6413
clean_incidence        1123
      clean_irs         243
      clean_itn         370
clean_mortality        1122
    clean_urban        6817
        raw_act         505
        raw_gdp         266
  raw_incidence        2747
        raw_irs         579
        raw_itn         400
  raw_mortality        2834
      raw_urban         266
regression_data         371
     trend_data        1123


In [8]:
# Preview first 3 rows of each table
# Confirms data loaded correctly and columns look right

tables_to_preview = [
    'raw_mortality',
    'raw_incidence', 
    'raw_itn',
    'raw_act',
    'raw_irs',
    'raw_gdp',
    'raw_urban'
]

for table in tables_to_preview:
    print(f"\n{'='*60}")
    print(f"Table: {table}")
    print('='*60)
    df = run_query(f"SELECT * FROM {table} LIMIT 3;", engine)
    print(df.to_string(index=False))


Table: raw_mortality
     Id         IndicatorCode SpatialDimension SpatialDimensionValueCode ParentLocationCode ParentLocation TimeDimension  TimeDim DisaggregatingDimension1 DisaggregatingDimension1ValueCode DisaggregatingDimension2 DisaggregatingDimension2ValueCode DisaggregatingDimension3 DisaggregatingDimension3ValueCode DataSourceDimension DataSourceDimensionValueCode               Value  NumericValue       Low      High Comments                 Date  TimeDimensionValue TimeDimensionBegin TimeDimensionEnd Unnamed: 25 Unnamed: 26
9418354 MALARIA_EST_MORTALITY          COUNTRY                       ECU                AMR       Americas          YEAR     2010                     None                              None                     None                              None                     None                              None                None                         None             0 [0-0]      0.000000  0.000000  0.000000     None 2025-12-19T14:27:58Z                201

## 4. Data Cleaning & Transformation

### Approach
- Filter all WHO files to African region (ParentLocationCode = 'AFR')
- Handle IRS missing region codes for BDI, SEN, ZWE, KEN separately
- Select only relevant columns from each table
- Convert IRS and ACT counts to rates per 100,000 population
- Unpivot World Bank data from wide to long format
- Filter World Bank data to 2000–2024

### Issues identified in audit
| Issue | Affected File | Fix |
|-------|--------------|-----|
| Zeros in non-African countries | Mortality, Incidence | AFR filter removes them |
| Missing region codes | IRS — BDI, SEN, ZWE, KEN | Filter by country code list |
| Counts not rates | IRS, ACT | Converted to binary – see Section 5.2 |
| Wide format | World Bank GDP, Urban | Unpivot in SQL |

### 4.2 Clean WHO Burden Tables

Filters for the African region and selects relevant columns.
Incidence is retained at per 1,000 population at risk (WHO standard).
Mortality rate per 100,000 population (WHO standard).
Different denominators reflect WHO reporting conventions — always analysed separately.

### 4.1 Define Sub-Saharan Africa Country List

Using WHO African region (AFR) as the geographic scope.
Four countries missing region codes in IRS file are included
explicitly by country code to prevent data loss.

In [9]:
# Sub-Saharan African country codes — WHO African region
# Used as the primary filter across all datasets
# Sourced from WHO GHO ParentLocationCode = 'AFR'

SSA_REGION_CODE = 'AFR'

# Countries missing ParentLocationCode in IRS file
# Identified during audit — all confirmed African nations
# Included explicitly to prevent accidental exclusion
IRS_MISSING_REGION = ('BDI', 'SEN', 'ZWE', 'KEN')

print(f"Primary region filter : {SSA_REGION_CODE}")
print(f"IRS explicit includes : {IRS_MISSING_REGION}")

Primary region filter : AFR
IRS explicit includes : ('BDI', 'SEN', 'ZWE', 'KEN')


In [10]:
# Clean mortality table
# Filter to the African region, rename columns for clarity

mortality_clean = """
    CREATE TABLE clean_mortality AS
    SELECT
        SpatialDimensionValueCode   AS country_code,
        ParentLocationCode          AS region,
        TimeDim                     AS year,
        NumericValue                AS mortality_per_100k,
        Low                         AS mortality_low,
        High                        AS mortality_high
    FROM raw_mortality
    WHERE ParentLocationCode = 'AFR'
    AND   TimeDim BETWEEN 2000 AND 2024;
"""

# Clean incidence table
# Retained at per 1,000 population at risk — WHO reporting standard
# Analysed separately from mortality — different denominators

incidence_clean = """
    CREATE TABLE clean_incidence AS
    SELECT
        SpatialDimensionValueCode       AS country_code,
        ParentLocationCode              AS region,
        TimeDim                         AS year,
        NumericValue                    AS incidence_per_1000,
        Low                             AS incidence_low,
        High                            AS incidence_high
    FROM raw_incidence
    WHERE ParentLocationCode = 'AFR'
    AND   TimeDim BETWEEN 2000 AND 2024;
"""

def create_clean_table(query, table_name, engine):
    """
    Drops table if exists then recreates it.
    Safe to rerun without manual cleanup.
    """
    try:
        with engine.connect() as conn:
            # Drop if exists — allows clean reruns
            conn.execute(text(f"DROP TABLE IF EXISTS {table_name};"))
            conn.execute(text(query))
            conn.commit()

            # Confirm row count
            result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};"))
            count = result.fetchone()[0]
            print(f"✓ {table_name:<25} {count:>5} rows")

    except Exception as e:
        print(f"✗ {table_name} failed: {e}")


# Create both clean burden tables
print("Creating clean burden tables...\n")
create_clean_table(mortality_clean,  'clean_mortality',  engine)
create_clean_table(incidence_clean,  'clean_incidence',  engine)

Creating clean burden tables...

✓ clean_mortality            1122 rows
✓ clean_incidence            1123 rows


### 4.3 Clean WHO Intervention Tables

ITN coverage is already a percentage — used directly as main predictor.
IRS and ACT loaded as raw counts here.
Both converted to binary variables in regression_data build step
due to high missingness (48% and 37% respectively) and absence
of population denominators. See Section 5.2 and audit log.

In [11]:
# Clean ITN coverage table
# Already a percentage — no transformation needed

itn_clean = """
    CREATE TABLE clean_itn AS
    SELECT
        SpatialDimensionValueCode   AS country_code,
        ParentLocationCode          AS region,
        TimeDim                     AS year,
        NumericValue                AS itn_coverage_pct
    FROM raw_itn
    WHERE ParentLocationCode = 'AFR'
    AND   TimeDim BETWEEN 2015 AND 2024;
"""

# Clean ACT treated table
# Raw count — converted to binary in regression_data build step
# See Section 5.2 for decision rationale

act_clean = """
    CREATE TABLE clean_act AS
    SELECT
        SpatialDimensionValueCode   AS country_code,
        ParentLocationCode          AS region,
        TimeDim                     AS year,
        NumericValue                AS act_treated_count
    FROM raw_act
    WHERE ParentLocationCode = 'AFR'
    AND   TimeDim BETWEEN 2015 AND 2024;
"""

# Clean IRS coverage table
# Four African countries missing region codes — included explicitly
# BDI=Burundi, SEN=Senegal, ZWE=Zimbabwe, KEN=Kenya

irs_clean = """
    CREATE TABLE clean_irs AS
    SELECT
        SpatialDimensionValueCode   AS country_code,
        ParentLocationCode          AS region,
        TimeDim                     AS year,
        NumericValue                AS irs_protected_count
    FROM raw_irs
    WHERE (
        ParentLocationCode = 'AFR'
        OR SpatialDimensionValueCode IN ('BDI', 'SEN', 'ZWE', 'KEN')
    )
    AND TimeDim BETWEEN 2015 AND 2024;
"""

# Create all three intervention tables
print("Creating clean intervention tables...\n")
create_clean_table(itn_clean, 'clean_itn', engine)
create_clean_table(act_clean, 'clean_act', engine)
create_clean_table(irs_clean, 'clean_irs', engine)

Creating clean intervention tables...

✓ clean_itn                   370 rows
✓ clean_act                   271 rows
✓ clean_irs                   243 rows


### 4.4 Transform World Bank Data — Wide to Long

World Bank data is downloaded in wide format with years as columns.
Unpivoted to long format to match WHO data structure.
Filtered to African countries and 2000–2024 only.
Join key: Country Code + Year — matches WHO SpatialDimensionValueCode + TimeDim.

In [12]:
# World Bank data arrives in wide format — one row per country
# years spread across 65 columns (1960-2024)
# Unpivot to long format: one row per country per year
# This matches the structure of all WHO tables

def unpivot_worldbank(table_name, value_column_name, engine):
    """
    Reads a World Bank wide-format table from MySQL,
    unpivots year columns to long format,
    filters to African countries and 2000-2024,
    returns a clean long-format DataFrame.
    """
    # Load full wide table into pandas
    df = run_query(f"SELECT * FROM {table_name};", engine)

    # Identify year columns — they are numeric strings
    # Non-year columns are metadata
    id_cols = ['Country Name', 'Country Code', 
               'Indicator Name', 'Indicator Code']
    
    year_cols = [col for col in df.columns 
                 if col.isdigit()]

    # Unpivot — melt year columns into rows
    df_long = df.melt(
        id_vars    = id_cols,
        value_vars = year_cols,
        var_name   = 'year',
        value_name = value_column_name
    )

    # Convert year and value to numeric
    df_long['year']              = pd.to_numeric(df_long['year'])
    df_long[value_column_name]   = pd.to_numeric(
                                        df_long[value_column_name], 
                                        errors='coerce'
                                    )

    # Filter to 2000-2024 only
    df_long = df_long[
        (df_long['year'] >= 2000) & 
        (df_long['year'] <= 2024)
    ]

    # Keep only relevant columns and rename for consistency
    df_long = df_long[['Country Code', 'year', value_column_name]]
    df_long.columns = ['country_code', 'year', value_column_name]

    # Drop rows with missing values
    # World Bank has gaps for small territories
    df_long = df_long.dropna(subset=[value_column_name])

    return df_long


# Unpivot both World Bank tables
print("Unpivoting World Bank tables...\n")

gdp_long   = unpivot_worldbank('raw_gdp',   'gdp_per_capita', engine)
urban_long = unpivot_worldbank('raw_urban', 'urban_pct',      engine)

print(f"✓ gdp_long     {len(gdp_long):>6} rows")
print(f"✓ urban_long   {len(urban_long):>6} rows")
print(f"\nGDP sample:")
print(gdp_long.head())

Unpivoting World Bank tables...

✓ gdp_long       6433 rows
✓ urban_long     6625 rows

GDP sample:
      country_code  year  gdp_per_capita
10640          ABW  2000    20681.023030
10641          AFE  2000      706.727261
10642          AFG  2000      174.930991
10643          AFW  2000      518.969226
10644          AGO  2000      563.733796


In [13]:
# Write unpivoted World Bank tables to MySQL
# Stored as clean tables alongside WHO clean tables

def write_df_to_mysql(df, table_name, engine):
    """
    Writes a pandas DataFrame to MySQL.
    Replaces table if it already exists.
    """
    try:
        df.to_sql(
            name      = table_name,
            con       = engine,
            if_exists = 'replace',
            index     = False
        )
        print(f"✓ {table_name:<25} {len(df):>6} rows written")
    except Exception as e:
        print(f"✗ {table_name} failed: {e}")


print("Writing World Bank tables to MySQL...\n")
write_df_to_mysql(gdp_long,   'clean_gdp',   engine)
write_df_to_mysql(urban_long, 'clean_urban', engine)

Writing World Bank tables to MySQL...

✓ clean_gdp                   6433 rows written
✓ clean_urban                 6625 rows written


## 5. Build Analytical Tables

Two analytical tables serve different analytical purposes:

**trend_data** — 2000 to 2024
Used for time series analysis and visualisation.
Contains burden indicators and economic controls only.
Wider year range captures full 24-year malaria trajectory.

**regression_data** — 2015 to 2024
Used for regression, machine learning, and forecasting.
Contains all variables including intervention coverage.
Year range constrained by ITN/IRS/ACT data availability.

### 5.1 Build trend_data (2000–2024)

In [14]:
# trend_data joins burden indicators with economic controls
# Covers full 2000-2024 period for time series analysis
# One row per country per year
# Africa only — filtered via WHO region code

trend_data_query = """
    CREATE TABLE trend_data AS
    SELECT
        i.country_code,
        i.region,
        i.year,

        -- Burden indicators
        -- Incidence retained as per 1,000 population at risk
        -- Mortality retained as per 100,000 population
        -- Different denominators reflect WHO reporting standards
        -- Always analysed on separate axes
        i.incidence_per_1000,
        i.incidence_low,
        i.incidence_high,
        m.mortality_per_100k,
        m.mortality_low,
        m.mortality_high,

        -- Economic and demographic controls
        g.gdp_per_capita,
        u.urban_pct

    FROM clean_incidence i

    LEFT JOIN clean_mortality m
        ON  i.country_code = m.country_code
        AND i.year         = m.year

    LEFT JOIN clean_gdp g
        ON  i.country_code = g.country_code
        AND i.year         = g.year

    LEFT JOIN clean_urban u
        ON  i.country_code = u.country_code
        AND i.year         = u.year

    ORDER BY i.country_code, i.year;
"""

print("Building trend_data...\n")
create_clean_table(trend_data_query, 'trend_data', engine)

Building trend_data...

✓ trend_data                 1123 rows


In [15]:
# Inspect trend_data
# Check structure, sample rows, and missing value counts

print("=== trend_data overview ===\n")

# Row count
shape = run_query("SELECT COUNT(*) AS row_count FROM trend_data;", engine)
print(f"Rows: {shape['row_count'].values[0]}")

# Sample rows
print("\nSample rows:")
sample = run_query("""
    SELECT * FROM trend_data 
    ORDER BY country_code, year 
    LIMIT 5;
""", engine)
print(sample.to_string(index=False))

# Missing value counts per column
print("\nMissing values per column:")
missing = run_query("""
    SELECT
        SUM(CASE WHEN country_code       IS NULL THEN 1 ELSE 0 END) AS country_code,
        SUM(CASE WHEN year               IS NULL THEN 1 ELSE 0 END) AS year,
        SUM(CASE WHEN incidence_per_1000 IS NULL THEN 1 ELSE 0 END) AS incidence,
        SUM(CASE WHEN mortality_per_100k IS NULL THEN 1 ELSE 0 END) AS mortality,
        SUM(CASE WHEN gdp_per_capita     IS NULL THEN 1 ELSE 0 END) AS gdp,
        SUM(CASE WHEN urban_pct          IS NULL THEN 1 ELSE 0 END) AS urban
    FROM trend_data;
""", engine)
print(missing.to_string(index=False))

# Year range and country count
print("\nCoverage summary:")
coverage = run_query("""
    SELECT 
        MIN(year)                    AS first_year,
        MAX(year)                    AS last_year,
        COUNT(DISTINCT country_code) AS country_count,
        COUNT(DISTINCT year)         AS year_count
    FROM trend_data;
""", engine)
print(coverage.to_string(index=False))

=== trend_data overview ===

Rows: 1123

Sample rows:
country_code region  year  incidence_per_1000  incidence_low  incidence_high  mortality_per_100k  mortality_low  mortality_high  gdp_per_capita  urban_pct
         AGO    AFR  2000          326.539432     224.192244      457.138368          138.099304     124.631920      154.407094      563.733796  50.507065
         AGO    AFR  2001          330.804593     223.501228      474.320442          140.674203     126.415109      158.068148      533.586202  51.722479
         AGO    AFR  2002          315.697197     214.332640      455.721125          125.181076     112.265339      140.982366      999.065856  52.892623
         AGO    AFR  2003          319.287503     221.664606      445.100493          121.725092     108.478112      137.758564     1133.663320  54.004355
         AGO    AFR  2004          312.646868     220.431475      432.513564          114.078051     100.820153      129.728367     1451.471179  55.044528

Missing values 

### 5.2 Build regression_data (2015–2024)

Joins all seven cleaned tables into one analytical table.
Restricted to 2015–2024 where intervention data is available.
Mayotte excluded — missing economic controls for entire period.
IRS and ACT converted from raw counts to rates per 100,000
using GDP table population proxy.

One row = one country, one year, all variables present.
This table is the direct input for:
- Multiple linear regression (SPSS)
- Random Forest prediction model (Python ML)
- Prophet forecasting model (Python ML)

In [16]:
# Build regression_data — primary modelling table (2015–2024)
#
# Scope: WHO African region, active malaria transmission countries only
# IRS and ACT converted to binary — high missingness, no population denominator
# ITN retained as continuous % — main predictor
# Full decisions documented in /docs/data_audit_log.txt

regression_data_query = """
    CREATE TABLE regression_data AS
    SELECT
        i.country_code,
        i.region,
        i.year,

        -- Outcomes
        i.incidence_per_1000,
        m.mortality_per_100k,

        -- Main predictor
        itn.itn_coverage_pct,

        -- Intervention controls (binary — see audit log)
        CASE 
            WHEN irs.irs_protected_count IS NOT NULL 
            AND  irs.irs_protected_count > 0 
            THEN 1 ELSE 0 
        END AS irs_deployed,

        CASE 
            WHEN act.act_treated_count IS NOT NULL 
            AND  act.act_treated_count > 0 
            THEN 1 ELSE 0 
        END AS act_deployed,

        -- Economic controls
        g.gdp_per_capita,
        u.urban_pct

    FROM clean_incidence i

    LEFT JOIN clean_mortality m
        ON  i.country_code = m.country_code
        AND i.year         = m.year

    LEFT JOIN clean_itn itn
        ON  i.country_code = itn.country_code
        AND i.year         = itn.year

    LEFT JOIN clean_irs irs
        ON  i.country_code = irs.country_code
        AND i.year         = irs.year

    LEFT JOIN clean_act act
        ON  i.country_code = act.country_code
        AND i.year         = act.year

    LEFT JOIN clean_gdp g
        ON  i.country_code = g.country_code
        AND i.year         = g.year

    LEFT JOIN clean_urban u
        ON  i.country_code = u.country_code
        AND i.year         = u.year

    WHERE i.year BETWEEN 2015 AND 2024
    AND   i.country_code != 'MYT'
    AND   i.country_code NOT IN (
              'BWA', 'CPV', 'DZA', 'NAM',
              'STP', 'SWZ', 'ZAF'
          )

    ORDER BY i.country_code, i.year;
"""

print("Building regression_data...\n")
create_clean_table(regression_data_query, 'regression_data', engine)

Building regression_data...

✓ regression_data             371 rows


In [17]:
# Inspect regression_data
# Verify structure, sample rows, coverage and missing values

print("=== regression_data overview ===\n")

# Sample rows
print("Sample rows:")
sample = run_query("""
    SELECT * FROM regression_data 
    ORDER BY country_code, year 
    LIMIT 5;
""", engine)
print(sample.to_string(index=False))

# Missing value counts per column
print("\nMissing values per column:")
missing = run_query("""
    SELECT
        SUM(CASE WHEN country_code       IS NULL THEN 1 ELSE 0 END) AS country_code,
        SUM(CASE WHEN year               IS NULL THEN 1 ELSE 0 END) AS year,
        SUM(CASE WHEN incidence_per_1000 IS NULL THEN 1 ELSE 0 END) AS incidence,
        SUM(CASE WHEN mortality_per_100k IS NULL THEN 1 ELSE 0 END) AS mortality,
        SUM(CASE WHEN itn_coverage_pct   IS NULL THEN 1 ELSE 0 END) AS itn,
        SUM(CASE WHEN irs_deployed       IS NULL THEN 1 ELSE 0 END) AS irs,
        SUM(CASE WHEN act_deployed       IS NULL THEN 1 ELSE 0 END) AS act,
        SUM(CASE WHEN gdp_per_capita     IS NULL THEN 1 ELSE 0 END) AS gdp,
        SUM(CASE WHEN urban_pct          IS NULL THEN 1 ELSE 0 END) AS urban
    FROM regression_data;
""", engine)
print(missing.to_string(index=False))

# Coverage summary
print("\nCoverage summary:")
coverage = run_query("""
    SELECT 
        MIN(year)                    AS first_year,
        MAX(year)                    AS last_year,
        COUNT(DISTINCT country_code) AS country_count,
        COUNT(DISTINCT year)         AS year_count
    FROM regression_data;
""", engine)
print(coverage.to_string(index=False))

=== regression_data overview ===

Sample rows:
country_code region  year  incidence_per_1000  mortality_per_100k  itn_coverage_pct  irs_deployed  act_deployed  gdp_per_capita  urban_pct
         AGO    AFR  2015          174.471775           42.936596         21.291318             0             0     3641.728939  63.515678
         AGO    AFR  2016          189.779622           44.412737         22.390819             0             0     2051.814621  64.299230
         AGO    AFR  2017          215.314558           45.804775         29.665611             0             0     2790.718869  65.085053
         AGO    AFR  2018          232.442981           46.007377         33.294117             0             1     2860.093648  65.873214
         AGO    AFR  2019          243.103177           47.112594         42.116596             0             1     2493.678844  66.663782

Missing values per column:
 country_code  year  incidence  mortality  itn  irs  act  gdp  urban
          0.0   0.0   

## 6. Export Analytical Tables

Exports trend_data and regression_data to CSV.
These files are the direct inputs for:
- Notebook 02: Exploratory analysis
- Notebook 03: Regression analysis (also exported to SPSS)
- Notebook 04: Machine learning models

In [18]:
# Export analytical tables to /data/cleaned/
# These are the analysis-ready outputs of the entire SQL pipeline

CLEANED_DATA_PATH = os.path.join(
    os.path.dirname(os.getcwd()), 'data', 'cleaned'
)

def export_table_to_csv(table_name, filename, data_path, engine):
    """
    Reads a MySQL table and exports it as CSV
    to the cleaned data folder.
    """
    try:
        df = run_query(f"SELECT * FROM {table_name};", engine)
        filepath = os.path.join(data_path, filename)
        df.to_csv(filepath, index=False)
        print(f"✓ {table_name:<25} exported → {filename}  ({len(df)} rows)")
        return df
    except Exception as e:
        print(f"✗ {table_name} export failed: {e}")
        return None


print("Exporting analytical tables...\n")

trend_df      = export_table_to_csv(
                    'trend_data',
                    'trend_data.csv',
                    CLEANED_DATA_PATH,
                    engine
                )

regression_df = export_table_to_csv(
                    'regression_data',
                    'regression_data.csv',
                    CLEANED_DATA_PATH,
                    engine
                )

print("\nExport complete.")
print(f"Files saved to: {CLEANED_DATA_PATH}")

Exporting analytical tables...

✓ trend_data                exported → trend_data.csv  (1123 rows)
✓ regression_data           exported → regression_data.csv  (371 rows)

Export complete.
Files saved to: C:\Users\HP\Desktop\malaria_analysis\data\cleaned
